# 00 · Normas e referências de segurança OT

**Objetivo:** antes de mexer em rede, entender o mapa das referências oficiais, o que cada uma cobre e como elas se encaixam. Os exercícios 01 a 07 aplicam estas ideias numa planta fictícia.

> Os textos normativos da IEC 62443 são vendidos pela IEC e pela ISA e não são reproduzidos aqui. Este repositório cita os identificadores dos requisitos (por exemplo *SR 5.1*) e explica com palavras próprias. NIST e MITRE publicam seus documentos gratuitamente.

## 1. O mapa

| Referência | Quem publica | Pergunta que responde | Onde aparece no lab |
|---|---|---|---|
| **IEC 62443** (série) | IEC / ISA | Como projetar, operar e manter um sistema de automação seguro? | zonas, SL, requisitos (02, 03, 07) |
| **NIST SP 800-82 Rev. 3** | NIST (EUA), 2023 | Guia prático de segurança para OT, com arquiteturas e controles | segmentação, DMZ, monitoramento (04, 06) |
| **NIST CSF 2.0** | NIST, 2024 | Como organizar um programa de cibersegurança em funções | matriz de conformidade (07) |
| **MITRE ATT&CK for ICS** | MITRE | O que atacantes realmente fazem em ambientes industriais? | ameaças e detecções (03, 06) |
| **Modelo Purdue** (ISA-95 / PERA) | indústria | Em que camadas está dividida a rede de uma fábrica? | desenho das zonas (02) |
| **ANEEL REN 964/2021 e rotina RO-CB.BR.01 do ONS** | reguladores brasileiros | Que controles mínimos o setor elétrico precisa ter? | contexto: mesma lógica de inventário, segmentação, monitoramento |

## 2. A série IEC 62443 em uma tabela

A série é dividida por **quem** precisa cumprir cada parte:

| Grupo | Partes mais citadas | Para quem | Em uma frase |
|---|---|---|---|
| 1 · Geral | 1-1 | todos | conceitos, termos, modelos |
| 2 · Políticas e procedimentos | 2-1, 2-3, 2-4 | dono do ativo, prestadores | programa de segurança, gestão de patches, requisitos para integradores |
| 3 · Sistema | **3-2**, **3-3** | integrador e dono do ativo | **3-2**: avaliar risco, dividir em zonas e conduítes, definir SL-T. **3-3**: requisitos de sistema (SR) por nível |
| 4 · Componente | 4-1, 4-2 | fabricantes | ciclo de desenvolvimento seguro e requisitos de produto |

Este lab trabalha principalmente com **3-2** (exercícios 02 e 03) e **3-3** (exercícios 04 a 07).

In [1]:
# Configuração: raiz do repositório no caminho de import, tabelas largas
import sys, pathlib, logging, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)
from IPython.display import Image, Markdown, display
from otsec.model import CAPTURES
from otsec.model import FR, FR_NAMES, requirements

## 3. Os 7 requisitos fundamentais (FR)

Toda a 62443-3-3 se organiza em sete FR. Cada requisito de sistema (SR) pertence a um deles.

In [2]:
reqs = pd.DataFrame(requirements()["requirements"])
fr = pd.DataFrame({"FR": FR, "nome": [FR_NAMES[f] for f in FR]})
fr["SR usados neste lab"] = fr.FR.map(reqs.groupby("fr").id.apply(lambda s: ", ".join(s)))
fr

,FR,nome,SR usados neste lab
0,IAC,Identification and authentication control,"SR 1.1, SR 1.2, SR 1.5, SR 1.7, SR 1.8, SR 1.13"
1,UC,Use control,"SR 2.1, SR 2.8"
2,SI,System integrity,"SR 3.1, SR 3.2"
3,DC,Data confidentiality,"SR 4.1, SR 4.3"
4,RDF,Restricted data flow,"SR 5.1, SR 5.2"
5,TRE,Timely response to events,"SR 6.1, SR 6.2"
6,RA,Resource availability,"SR 7.1, SR 7.3, SR 7.8"


**Leitura:** RDF (*Restricted data flow*) é onde mora a segmentação; TRE (*Timely response to events*) é monitoramento e resposta; RA (*Resource availability*) lembra que em OT a **disponibilidade** costuma vir antes da confidencialidade. Uma linha parada custa mais que um dado vazado na maioria das fábricas.

## 4. Níveis de segurança (SL)

O SL descreve **contra quem** a zona precisa resistir. Resumo com palavras próprias:

| SL | Resiste a | Exemplo |
|---|---|---|
| 1 | violação casual ou acidental | operador que conecta um pendrive por engano |
| 2 | ataque intencional com meios simples, poucos recursos, conhecimento genérico | ransomware comum que chega pela rede de TI |
| 3 | ataque sofisticado, recursos moderados, conhecimento específico de automação | grupo criminoso que estuda o protocolo do CLP |
| 4 | ataque sofisticado com muitos recursos e alta motivação | ator patrocinado por Estado |

Três siglas que aparecem o tempo todo:

* **SL-T** (*target*): o nível que a zona **precisa** ter, saído da análise de risco.
* **SL-C** (*capability*): o nível que um componente **consegue** entregar se bem configurado.
* **SL-A** (*achieved*): o nível que a zona **tem hoje**, com os controles implementados.

O trabalho de segurança é fechar a distância entre SL-A e SL-T. O exercício 03 mede isso.

## 5. Como o lab percorre o ciclo

```
 01 inventário  ->  02 zonas e conduítes  ->  03 risco e SL-T
                                                  |
 07 conformidade <- 06 detecção <- 05 hardening <- 04 segmentação
```

É a ordem que a 62443-3-2 sugere: **conhecer** o sistema, **dividir**, **avaliar**, **proteger**, **detectar** e **provar**. O NIST CSF 2.0 chama essas etapas de *Identify, Protect, Detect* (e acrescenta *Govern, Respond, Recover*).

In [3]:
from otsec import risk
c = risk.controls()
print("Controles do lab:", len(c), "| implementados:", (c.status == "implemented").sum(), "| planejados:", (c.status == "planned").sum())
c[["id", "name", "exercise", "status"]]

Controles do lab: 13 | implementados: 9 | planejados: 4


,id,name,exercise,status
0,CTL-INV,Passive asset inventory from network captures,01,implemented
1,CTL-ZONES,Zone and conduit model with design rules,02,implemented
2,CTL-FW,Default-deny firewall between zones generated from the conduits,04,implemented
3,CTL-DMZ,Enterprise reaches plant data only through the DMZ broker,04,implemented
4,CTL-MQTT-TLS,MQTT over TLS with a private CA,05,implemented
5,CTL-MQTT-AUTH,"MQTT without anonymous access, one account per client, strong passwords",05,implemented
6,CTL-MQTT-ACL,Per-topic ACL on the UNS broker,05,implemented
7,CTL-OPCUA-SEC,"OPC UA with SignAndEncrypt, application certificates and user login",05,implemented
8,CTL-IDS,"Modbus allowlist IDS (talkers, function codes, write ranges, rates)",06,implemented
9,CTL-MFA,MFA and a jump host for remote access,-,planned


## Resumo

* **62443** é a referência de projeto para automação: zonas, conduítes, níveis de segurança e requisitos por FR.
* **NIST 800-82** traduz isso em arquitetura prática; **CSF 2.0** organiza o programa; **ATT&CK for ICS** dá nome às ameaças.
* O lab trata cada controle como algo **testável**. No exercício 07, um requisito só conta como atendido se os testes automáticos passarem.